In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    precision_recall_curve
)

import optuna


In [ ]:
# Load the dataset
df = pd.read_csv("data/creditcard.csv")
df.info()
df.head()
df.shape

## Exploratory Data Analysis

In [ ]:
# Count of fraud vs non-fraud
print(df['Class'].value_counts())

sns.countplot(x='Class', data=df)
plt.title("Fraud vs Non-Fraud Transactions")
plt.show()

# Percentage
print(df['Class'].value_counts(normalize=True) * 100)

In [ ]:
# Correlation heatmap
plt.figure(figsize=(14,10))
sns.heatmap(df.corr(), cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

## Data Preprocessing

In [5]:
# Feature scaling for 'Amount' and 'Time'
scaler = StandardScaler()

df['scaled_amount'] = scaler.fit_transform(df[['Amount']])
df['scaled_time'] = scaler.fit_transform(df[['Time']])

In [ ]:
# Drop original columns
df = df.drop(['Amount', 'Time'], axis=1)

In [9]:
# Rearranging columns
cols = [col for col in df.columns if col != 'Class'] + ['Class']
df = df[cols]

## Model Training

In [10]:
# Define features and target
X = df.drop('Class', axis=1)
y = df['Class']

In [11]:
# Train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [12]:
#Applying SMOTE
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000, class_weight='balanced')
lr.fit(X_train_res, y_train_res)

In [ ]:
# Random Forest
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_res, y_train_res)

In [ ]:
# XGBoost
xgb = XGBClassifier(eval_metric='logloss', random_state=42)
xgb.fit(X_train_res, y_train_res)

In [ ]:
# Hyperparameter Tuning (GridSearchCV – Random Forest)
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [8, 10],
    'min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_params,
    cv=3,
    scoring='recall',
    n_jobs=-1
)

grid_rf.fit(X_train_res, y_train_res)

best_rf = grid_rf.best_estimator_
print("Best RF Params:", grid_rf.best_params_)

In [17]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.metrics import recall_score
import optuna

def objective(trial):
    
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'eval_metric': 'logloss',
        'random_state': 42
    }

    # Step 1: Split ORIGINAL training data
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train,
        test_size=0.2,
        random_state=42,
        stratify=y_train
    )

    #  Step 2: Apply SMOTE ONLY on training part
    smote = SMOTE(random_state=42)
    X_tr_res, y_tr_res = smote.fit_resample(X_tr, y_tr)

    #  Step 3: Train model
    model = XGBClassifier(**params)
    model.fit(X_tr_res, y_tr_res)

    #  Step 4: Evaluate on REAL validation data (no SMOTE)
    preds = model.predict(X_val)

    return recall_score(y_val, preds)

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

In [19]:
# Evaluation Function
def evaluate(model, X_test, y_test, threshold=0.5):
    y_prob = model.predict_proba(X_test)[:,1]
    y_pred = (y_prob > threshold).astype(int)

    print("\nThreshold:", threshold)
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_prob))

In [ ]:
# Compare Models
print("Logistic Regression")
evaluate(lr, X_test, y_test)

print("\nTuned Random Forest")
evaluate(best_rf, X_test, y_test)

print("\nTuned XGBoost")
evaluate(xgb, X_test, y_test)

In [ ]:
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# Apply SMOTE on full training data
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

best_xgb = XGBClassifier(
    n_estimators=129,
    max_depth=3,
    learning_rate=0.052,
    subsample=0.67,
    colsample_bytree=0.938,
    eval_metric='logloss',
    random_state=42
)

best_xgb.fit(X_train_res, y_train_res)

In [ ]:
# Threshold Tuning
xgb_prob = best_xgb.predict_proba(X_test)[:,1]

thresholds = np.arange(0.1, 0.6, 0.05)

print("Threshold | Precision | Recall")

for t in thresholds:
    preds = (xgb_prob > t).astype(int)
    
    precision = precision_score(y_test, preds)
    recall = recall_score(y_test, preds)
    
    print(f"{t:.2f}      | {precision:.3f}     | {recall:.3f}")

In [ ]:
# Threshold Tuning
xgb_prob = best_xgb.predict_proba(X_test)[:,1]

thresholds = np.arange(0.1, 0.6, 0.05)

print("Threshold | Precision | Recall")

for t in thresholds:
    preds = (xgb_prob > t).astype(int)
    
    precision = precision_score(y_test, preds)
    recall = recall_score(y_test, preds)
    
    print(f"{t:.2f}      | {precision:.3f}     | {recall:.3f}")

In [ ]:
# Final Model
best_threshold = 0.25  # adjust from above

final_pred = (xgb_prob > best_threshold).astype(int)

print("\n FINAL MODEL")
print(confusion_matrix(y_test, final_pred))
print(classification_report(y_test, final_pred))
print("ROC-AUC:", roc_auc_score(y_test, xgb_prob))

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve

y_prob = best_xgb.predict_proba(X_test)[:, 1]

# Find best threshold for recall
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

best_threshold = thresholds[np.argmax(recall >= 0.85)]
print("Best Threshold:", best_threshold)

y_pred = (y_prob > best_threshold).astype(int)

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

In [ ]:
from sklearn.metrics import precision_recall_curve
import numpy as np

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("Best Threshold:", best_threshold)

In [ ]:
y_pred = (y_prob > best_threshold).astype(int)

from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

In [ ]:
import os
import pickle

os.makedirs("models", exist_ok=True)

# Save trained model
with open("models/best_xgb.pkl", "wb") as file:
    pickle.dump(best_xgb, file)

# Save scaler
with open("models/scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

# Save threshold
with open("models/threshold.pkl", "wb") as file:
    pickle.dump(best_threshold, file)

print("Model and preprocessing files saved successfully.")